# Implementatie van het eindprogramma voor de Mystery Device

In [72]:
import numpy as np
np.random.seed(0)

## Definiëren van het model en functies

Voordat we het eindprogramma kunnen schrijven, moeten we natuurlijk eerst een model trainen. We hebben al verschillende modellen getraind in de P opdrachten, maar nu moeten we een model kiezen dat goed presteert en binnen de constraints van de Mystery Device past.

Zie `experimenten.ipynb` voor een overzicht van de gemaakte P opdrachten en de resultaten daarvan. Hier wordt ook de keuze voor het eindmodel gemotiveerd.

### Data laden

In [73]:
from tensorflow.keras.datasets import mnist

def load_mnist(npz_path='mnist.npz'):
    (x_train, y_train), (x_test, y_test) = mnist.load_data(path=npz_path)
    return (x_train, y_train), (x_test, y_test)

### Preprocessing

In [74]:
def normalize_data(x):
    return x / 255.0


def flatten_data(x):
    return x.reshape(len(x), -1)

### Model

In [75]:
epsilon = 1e-15


def relu(z):
    return np.maximum(0, z)


def relu_derivative(z):
    return (z > 0).astype(np.float32)


def softmax(z):
    z_shifted = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(z_shifted)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)


def one_hot(y, n_classes):
    return np.eye(n_classes)[y]


def sparse_cross_entropy(y_true, y_pred):
    probs = np.clip(y_pred, epsilon, 1.0 - epsilon)
    log_probs = np.log(probs)
    true_log_probs = log_probs[np.arange(len(y_true)), y_true]
    return -np.mean(true_log_probs)

In [76]:
class NeuralNetwork:
    def __init__(self, input_nodes, hidden_layer_sizes, output_nodes, dropout_rates=None):
        if dropout_rates is None:
            dropout_rates = [0.0] * len(hidden_layer_sizes)
        elif isinstance(dropout_rates, float):
            dropout_rates = [dropout_rates] * len(hidden_layer_sizes)
        self.dropout_rates = dropout_rates

        layer_sizes = [input_nodes] + hidden_layer_sizes + [output_nodes]
        self.weights = []
        self.biases = []
        self.w_min = []
        self.w_max = []

        for i in range(len(layer_sizes) - 1):
            w = np.random.randn(layer_sizes[i], layer_sizes[i + 1]) * np.sqrt(2 / layer_sizes[i])
            b = np.zeros((1, layer_sizes[i + 1]))
            self.weights.append(w)
            self.biases.append(b)

        self._best_val_loss = np.inf
        self._epochs_without_improvement = 0
        self._best_snapshot = self._snapshot()
    
    @classmethod
    def from_weights(cls, path, hidden_layer_sizes, output_nodes, dropout_rates=None):
        model = cls.__new__(cls)
        model.dropout_rates = dropout_rates or [0.0] * len(hidden_layer_sizes)
        model.weights = []
        model.biases = []
        model.w_min = []
        model.w_max = []
        model.load_weights(path)
        return model

    def _dequantize(self, i):
        return self.weights[i].astype(np.float32) / 255.0 * (self.w_max[i] - self.w_min[i]) + self.w_min[i]

    def _get_weight(self, i):
        if self.w_min:
            return self._dequantize(i)
        return self.weights[i]

    # Zorgt ervoor dat de volledige float32 matrix nooit volledig in RAM staat
    def _matmul(self, x, i):
        if not self.w_min:
            return x @ self.weights[i]

        W, w_min, w_max = self.weights[i], self.w_min[i], self.w_max[i]
        out = np.zeros((x.shape[0], W.shape[1]), dtype=np.float32)
        for j, col in enumerate(W.T):
            out[:, j] = x @ (col.astype(np.float32) / 255.0 * (w_max - w_min) + w_min)
        return out

    def forward(self, x, training=True):
        hidden_caches = []
        current = x

        for i in range(len(self.weights) - 1):
            preac = self._matmul(current, i) + self.biases[i]
            ac = relu(preac)

            mask = None
            if training:
                rate = self.dropout_rates[i]
                if rate > 0:
                    mask = (np.random.rand(*ac.shape) > rate) / (1 - rate)
                    ac = ac * mask
                hidden_caches.append((preac, ac, mask))
            current = ac

        o_preac = self._matmul(current, -1) + self.biases[-1]
        pred = softmax(o_preac)

        return x, hidden_caches, pred

    def backward(self, y_true, cache):
        x, hidden_caches, pred = cache
        n_hidden = len(hidden_caches)

        weight_gradients = [None] * len(self.weights)
        biases_gradients = [None] * len(self.biases)

        output_gradient = pred - one_hot(y_true, pred.shape[1])
        last_ac = hidden_caches[-1][1]
        weight_gradients[-1] = last_ac.T @ output_gradient
        biases_gradients[-1] = np.sum(output_gradient, axis=0, keepdims=True)

        gradient = output_gradient
        for i in reversed(range(n_hidden)):
            preac, ac, mask = hidden_caches[i]

            gradient = gradient @ self._get_weight(i + 1).T
            if mask is not None:
                gradient = gradient * mask
            gradient = gradient * relu_derivative(preac)

            prev_ac = hidden_caches[i - 1][1] if i > 0 else x
            weight_gradients[i] = prev_ac.T @ gradient
            biases_gradients[i] = np.sum(gradient, axis=0, keepdims=True)

        return weight_gradients, biases_gradients

    def _apply_gradients(self, weight_grads, bias_grads, learning_rate):
        for i in range(len(self.weights)):
            self.weights[i] -= learning_rate * weight_grads[i]
            self.biases[i] -= learning_rate * bias_grads[i]

    def _snapshot(self):
        return (
            [w.copy() for w in self.weights],
            [b.copy() for b in self.biases]
        )

    def _restore(self, snapshot):
        weights, biases = snapshot
        self.weights = [w.copy() for w in weights]
        self.biases = [b.copy() for b in biases]

    def train(self, x_train, y_train, learning_rate=0.001, epochs=50, batch_size=32, val_split=0.1, patience=12, verbose=True):
        from collections import deque

        n_val = int(len(x_train) * val_split)
        x_val, y_val = x_train[:n_val], y_train[:n_val]
        x_tr, y_tr = x_train[n_val:], y_train[n_val:]
        n = len(x_tr)

        best_info = None
        stopped_early = False

        for epoch in range(1, epochs + 1):
            indices = np.random.permutation(n)
            x_shuffled = x_tr[indices]
            y_shuffled = y_tr[indices]

            batch_queue = deque(
                (x_shuffled[start:start + batch_size], y_shuffled[start:start + batch_size])
                for start in range(0, n, batch_size)
            )

            epoch_loss = 0.0
            epoch_correct = 0

            while batch_queue:
                x_batch, y_batch = batch_queue.popleft()
                cache = self.forward(x_batch, training=True)

                _, _, pred = cache

                epoch_loss += sparse_cross_entropy(y_batch, pred) * len(x_batch)
                epoch_correct += np.sum(np.argmax(pred, axis=1) == y_batch)

                weight_grads, bias_grads = self.backward(y_batch, cache)
                self._apply_gradients(weight_grads, bias_grads, learning_rate)

            _, _, val_pred = self.forward(x_val, training=False)
            val_loss = sparse_cross_entropy(y_val, val_pred)

            if verbose:
                print(f"Epoch {epoch:3d} | "
                      f"Acc: {epoch_correct / n:.4f} | "
                      f"Loss: {epoch_loss / n:.4f} | "
                      f"Val Loss: {val_loss:.4f}")

            if val_loss < self._best_val_loss:
                self._best_val_loss = val_loss
                self._epochs_without_improvement = 0
                self._best_snapshot = self._snapshot()
                best_info = (epoch, epoch_correct / n, epoch_loss / n, val_loss)
            else:
                self._epochs_without_improvement += 1

            if self._epochs_without_improvement >= patience:
                stopped_early = True
                break

        self._restore(self._best_snapshot)
        if verbose:
            if stopped_early:
                print(f"\nEarly stopping na epoch {epoch}.")
            ep, acc, loss, vl = best_info
            print(f"Hersteld naar epoch {ep} | Acc: {acc:.4f} | Loss: {loss:.4f} | Val Loss: {vl:.4f}")

    def evaluate(self, x_test, y_test):
        _, _, pred = self.forward(x_test, training=False)
        loss = sparse_cross_entropy(y_test, pred)
        predicted = np.argmax(pred, axis=1)
        acc = np.mean(predicted == y_test)
        print(f"Test Accuracy: {acc:.4f} | Test Loss: {loss:.4f}")
        return acc, loss, predicted

    def predict(self, x):
        x = np.array(x, dtype=np.float32).reshape(1, -1)
        _, _, pred = self.forward(x, training=False)
        probabilities = pred[0]
        return int(np.argmax(probabilities)), probabilities

    # Opslaan en laden van gewichten met quantization
    def save_weights(self, path):
        arrays = {}
        for i, (weight, bias) in enumerate(zip(self.weights, self.biases)):
            for name, arr in [('w', weight), ('b', bias)]:
                arr_min, arr_max = arr.min(), arr.max()
                quantized = ((arr - arr_min) / (arr_max - arr_min) * 255).astype(np.uint8)
                arrays[f'{name}{i}'] = quantized
                arrays[f'{name}{i}_min'] = np.array([arr_min], dtype=np.float32)
                arrays[f'{name}{i}_max'] = np.array([arr_max], dtype=np.float32)
        np.savez_compressed(path, **arrays)

    def load_weights(self, path):
        data = np.load(path)
        self.weights = []
        self.biases = []
        self.w_min = []
        self.w_max = []
        i = 0

        while f'w{i}' in data:
            self.weights.append(data[f'w{i}'])
            self.w_min.append(data[f'w{i}_min'][0])
            self.w_max.append(data[f'w{i}_max'][0])

            b_min = data[f'b{i}_min'][0]
            b_max = data[f'b{i}_max'][0]
            b = data[f'b{i}'].astype(np.float32) / 255.0 * (b_max - b_min) + b_min
            self.biases.append(b)
            i += 1

## Trainen van het model

### Inladen en preprocessen van de data

In [77]:
(x_train, y_train), (x_test, y_test) = load_mnist()
x_train = normalize_data(flatten_data(x_train))
x_test = normalize_data(flatten_data(x_test))

### Trainen

In [78]:
input_nodes = 28 * 28
hidden_layer_sizes = [256, 128]
output_nodes = 10
dropout_rates = [0.2, 0.15]

model = NeuralNetwork(input_nodes, hidden_layer_sizes, output_nodes, dropout_rates)
model.train(x_train, y_train, learning_rate=0.003, epochs=200, batch_size=8, val_split=0.1, patience=20, verbose=True)

Epoch   1 | Acc: 0.9031 | Loss: 0.3241 | Val Loss: 0.1317
Epoch   2 | Acc: 0.9537 | Loss: 0.1527 | Val Loss: 0.0987
Epoch   3 | Acc: 0.9654 | Loss: 0.1150 | Val Loss: 0.0870
Epoch   4 | Acc: 0.9708 | Loss: 0.0946 | Val Loss: 0.0774
Epoch   5 | Acc: 0.9751 | Loss: 0.0777 | Val Loss: 0.0748
Epoch   6 | Acc: 0.9780 | Loss: 0.0686 | Val Loss: 0.0743
Epoch   7 | Acc: 0.9802 | Loss: 0.0622 | Val Loss: 0.0677
Epoch   8 | Acc: 0.9832 | Loss: 0.0539 | Val Loss: 0.0679
Epoch   9 | Acc: 0.9848 | Loss: 0.0486 | Val Loss: 0.0675
Epoch  10 | Acc: 0.9854 | Loss: 0.0461 | Val Loss: 0.0627
Epoch  11 | Acc: 0.9871 | Loss: 0.0402 | Val Loss: 0.0662
Epoch  12 | Acc: 0.9889 | Loss: 0.0344 | Val Loss: 0.0639
Epoch  13 | Acc: 0.9887 | Loss: 0.0331 | Val Loss: 0.0682
Epoch  14 | Acc: 0.9892 | Loss: 0.0318 | Val Loss: 0.0645
Epoch  15 | Acc: 0.9898 | Loss: 0.0301 | Val Loss: 0.0698
Epoch  16 | Acc: 0.9912 | Loss: 0.0266 | Val Loss: 0.0641
Epoch  17 | Acc: 0.9923 | Loss: 0.0245 | Val Loss: 0.0692
Epoch  18 | Ac

### Evalueren

In [79]:
model.evaluate(x_test, y_test)

Test Accuracy: 0.9824 | Test Loss: 0.0639


(np.float64(0.9824),
 np.float64(0.06389471853268354),
 array([7, 2, 1, ..., 4, 5, 6], shape=(10000,)))

### Opslaan van de gewichten

In [80]:
model.save_weights('weights.npz')

## Eindprogramma voor de Mystery Device

 ### Inladen van een afbeelding

In [81]:
from preprocess import ImagePreprocessor

In [82]:
preprocessor = ImagePreprocessor()

def load_image(filename):
    return preprocessor.process(filename)

### Classificeerfunctie

In [83]:
def classify_image(image, threshold=0.9):
    prediction, probabilities = mystery_model.predict(image)

    probs = probabilities.flatten()

    best_indices = np.argsort(probs)
    highest_idx = best_indices[-1]
    second_idx = best_indices[-2]

    highest_prob = probs[highest_idx]
    second_prob = probs[second_idx]

    if highest_prob < threshold:
        primary_pred = "niet herkend"
    else:
        primary_pred = str(highest_idx)

    guess_if_forced = str(highest_idx)

    return primary_pred, highest_prob, guess_if_forced, second_idx, second_prob

### Model maken op device

In [84]:
mystery_model = NeuralNetwork.from_weights('weights.npz', hidden_layer_sizes, output_nodes, dropout_rates)

### Testen van verschillende plaatjes

In [85]:
from pathlib import Path

png_files = sorted(Path(".").glob("*.png"))

header = f"{'Bestandsnaam':<13} | {'Verwacht':<12} | {'Voorspelling':<12} | {'Conf. #1':<8} | {'Gok (bij niet herkend)':<22} | {'Runner-up (#2)':<14}"

print(header)

print("-" * len(header))

for png_file in png_files:
    filename = png_file.name

    name_without_extension = png_file.stem
    if name_without_extension.isdigit():
        expected = name_without_extension
    else:
        expected = "niet herkend"

    image = load_image(filename)
    pred, conf1, guess, second_digit, conf2 = classify_image(image)

    if pred == "niet herkend":
        guess_str = guess
    else:
        guess_str = "-"

    runner_up_str = f"{second_digit} ({conf2:.2%})"

    print(
        f"{filename:<13} | {expected:<12} | {pred:<12} | {conf1:<8.2%} | {guess_str:<22} | {runner_up_str:<15}"
    )

Bestandsnaam  | Verwacht     | Voorspelling | Conf. #1 | Gok (bij niet herkend) | Runner-up (#2)
------------------------------------------------------------------------------------------------
0.png         | 0            | 0            | 100.00%  | -                      | 2 (0.00%)      
1.png         | 1            | 1            | 99.93%   | -                      | 8 (0.04%)      
2.png         | 2            | 2            | 99.93%   | -                      | 3 (0.06%)      
3.png         | 3            | 3            | 99.70%   | -                      | 5 (0.24%)      
4.png         | 4            | 4            | 99.97%   | -                      | 7 (0.02%)      
5.png         | 5            | 5            | 99.94%   | -                      | 9 (0.04%)      
6.png         | 6            | 6            | 97.02%   | -                      | 4 (2.81%)      
7.png         | 7            | 7            | 99.97%   | -                      | 2 (0.03%)      
8.png         | 8     

### Peak RAM tijdens inference meten

Tijdens inference zijn er meerdere sources die deel uitmaken van het uiteindelijke totaal peak RAM tijdens inference.
Hieronder volgen de verschillende arrays die samen de peak RAM maken.

- Persistente arrays: altijd in RAM
  - Model weights
  - Model biases
  - Min/max per laag
- Tijdelijke arrays tijdens een forward pass
  - Input laag
  - Pre-activation
  - Current
  - Gedequantiseerde gewichten per kolom per laag
  - Output pre-activation
  - Prediction

In [86]:
persistent_weights = sum(w.nbytes for w in mystery_model.weights)   # uint8
persistent_biases = sum(b.nbytes for b in mystery_model.biases)     # float32
persistent_minmax = len(mystery_model.w_min) * 2 * 4                # float32 min+max per laag

# _matmul dequantiseert kolom voor kolom; peak = één kolom van de grootste laag (laag 1: 784 floats)
peak_dequantized_col = input_nodes * 4

temporary = (
    1 * input_nodes * 4 +
    1 * hidden_layer_sizes[0] * 4 +                                 # preac laag 1
    1 * hidden_layer_sizes[0] * 4 +                                 # current laag 1
    peak_dequantized_col +                                          # dequantized gewichtenkolom (peak: laag 1)
    1 * hidden_layer_sizes[1] * 4 +                                 # preac laag 2
    1 * hidden_layer_sizes[1] * 4 +                                 # current laag 2
    1 * output_nodes * 4 +
    1 * output_nodes * 4
)

persistent = persistent_weights + persistent_biases + persistent_minmax
print(f"Gewichten (uint8):         {persistent_weights / 1024:.2f} KB")
print(f"Biases (float32):          {persistent_biases / 1024:.2f} KB")
print(f"Min/max (float32):         {persistent_minmax / 1024:.2f} KB")
print(f"Tijdelijke arrays:         {temporary / 1024:.2f} KB")
print(f"Totaal peak:               {(persistent + temporary) / 1024:.2f} KB")

Gewichten (uint8):         229.25 KB
Biases (float32):          1.54 KB
Min/max (float32):         0.02 KB
Tijdelijke arrays:         9.20 KB
Totaal peak:               240.02 KB


In [109]:
import tracemalloc

tracemalloc.start()

pre = tracemalloc.take_snapshot()
mystery_model = NeuralNetwork.from_weights('weights.npz', hidden_layer_sizes, output_nodes, dropout_rates)
snapshot_loaded = tracemalloc.take_snapshot()

mystery_model.predict(x_test[0])
snapshot_predicted = tracemalloc.take_snapshot()

tracemalloc.stop()

def total_kb(snapshot):
    return sum(stat.size for stat in snapshot.statistics('lineno')) / 1024

print(f"Begin runtime:   {total_kb(pre):.2f} KB")
print(f"Na maken model:  {total_kb(snapshot_loaded):.2f} KB")
print(f"Na predict:      {total_kb(snapshot_predicted):.2f} KB")

Begin runtime:   0.89 KB
Na maken model:  251.91 KB
Na predict:      252.43 KB
